# CSV Splitter - Simple Demo & Analysis

Simple notebook to run and analyze `split_multiclass_csv.py`

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd

sys.path.insert(0, os.path.abspath('..'))
from split_multiclass_csv import split_multiclass_csv, split_multiclass_csv_auto, process_multiclass_dataset

## 1. Configuration

In [ ]:
# Set your paths
INPUT_DIR = "path/to/input"
OUTPUT_DIR = "path/to/output"

# Mode: False = Q_ prefix mode, True = auto-detect mode
USE_AUTO_DETECT = False

print(f"Input:  {INPUT_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Mode:   {'Auto-detect' if USE_AUTO_DETECT else 'Q_ prefix'}")

## 2. Analyze Input

In [ ]:
input_csvs = list(Path(INPUT_DIR).rglob('*.csv'))
print(f"Found {len(input_csvs)} CSV file(s)\n")

# Show first CSV structure
if input_csvs:
    sample_csv = input_csvs[0]
    df = pd.read_csv(sample_csv)
    print(f"Sample: {sample_csv.name}")
    print(f"Columns: {list(df.columns)}")
    print(f"Rows: {len(df)}")
    print(f"\nFirst 3 rows:")
    print(df.head(3))

## 3. Run Splitter

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

if USE_AUTO_DETECT:
    # Auto-detect mode for CSVs without Q_ prefix
    base_cols = ['Filename', 'Starttime', 'Endtime']
    
    for csv_file in Path(INPUT_DIR).rglob('*.csv'):
        print(f"\nProcessing: {csv_file.name}")
        split_multiclass_csv_auto(csv_file, OUTPUT_DIR, base_columns=base_cols, verbose=True)
else:
    # Q_ prefix mode (standard)
    process_multiclass_dataset(
        input_dir=INPUT_DIR,
        output_dir=OUTPUT_DIR,
        copy_audio=True,
        copy_features=True,
        preserve_structure=True,
        create_symlinks=True,
        verbose=True
    )

## 4. Analyze Output

In [ ]:
output_csvs = list(Path(OUTPUT_DIR).rglob('*.csv'))
print(f"\n📊 Created {len(output_csvs)} output CSV file(s)\n")

# Count rows per file
total_rows = 0
for csv in output_csvs:
    df = pd.read_csv(csv)
    total_rows += len(df)
    print(f"{csv.name}: {len(df)} rows")

print(f"\nTotal POS labels: {total_rows}")

## 5. Simple Visualization

In [ ]:
import matplotlib.pyplot as plt

if output_csvs:
    # Count rows per output file
    names = [csv.stem for csv in output_csvs]
    counts = [len(pd.read_csv(csv)) for csv in output_csvs]
    
    plt.figure(figsize=(10, 6))
    plt.barh(names, counts, color='steelblue')
    plt.xlabel('Number of POS Labels')
    plt.title('POS Labels per Split CSV')
    plt.tight_layout()
    plt.show()
    
    print(f"\n✅ Done! Output saved to: {OUTPUT_DIR}")

## Summary

**Two modes available:**
- **Q_ prefix mode**: For CSVs with `Q_species1`, `Q_species2` columns
- **Auto-detect mode**: For CSVs with `species1`, `species2` columns (no Q_ prefix)

The splitter creates separate CSV files for each class, filtering only POS labels.